### 

In [ ]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

In [ ]:
from flask import Flask, request, jsonify
import threading
from sentence_transformers import SentenceTransformer
import json
import requests
from typing import Union
import os
from contextlib import redirect_stdout, redirect_stderr
import json
import requests
from typing import Union, List, Dict, Any
import os
from contextlib import redirect_stdout, redirect_stderr
import logging
from typing import Dict, List
import torch
import os
import subprocess
import signal
import sys
import logging
import httpx
from openai import OpenAI, DefaultHttpxClient

In [ ]:
########################################################################
#######################################################################
####################
###############################
####################################
#########################################
######################################################################

In [ ]:
import weaviate
from weaviate.classes.config import Configure

from utils import (
print_object_properties ,
suppress_weaviate_logs
)
import flask_app
from weaviate.classes.config import DataType ,Property ,Configure

In [ ]:
from sentence_transformers import sentence_transformer

In [ ]:
client.close()

In [ ]:
with suppress_weaviate_logs () :
    client=weaviate.connect_to_embedded(
        persistence_data_path="./.collections" ,
        environment_variables={
            "ENABLE_API_BASED_MODULES":"true" ,
            "ENABLE_MODULES":"text2vec-transformers ,reranker-transformers",
            "TRANSFORMERS_INFERENCE_API": "http://127.0.0.1:5000/", 
            "RERANKER_INFERENCE_API": "http://127.0.0.1:5000/"
        }
    )

In [ ]:
if client.collections.exists("example_collection"):
    client.collections.delete("example_collection")
    print ("Done")

In [ ]:
import joblib 

In [ ]:
data=joblib.load("data.joblib")

In [ ]:
print_object_properties(data[0])

In [ ]:
vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
    name="vector",
    source_properties=['place','state','description','best_season_to_visit','attractions','budget'],
    vectorize_collection_name=False ,
    inference_url="http://127.0.0.1:5000/"
    
) ]

In [ ]:
if not client.collections.exists("example_collection"):
    collection=client.collections.create(
        
        
        name="example_collection" ,
        vectorizer_config=vectorizer_config ,
        reranker_config=Configure.Reranker.transformers() ,
        
        properties=[
            
            Property(name="place" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="state" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="description" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="best_season_to_visit" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="attractions" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="budget" ,vectorize_property_name=True ,data_type=DataType.TEXT ) ,
            Property(name="user_ratings"  ,data_type=DataType.NUMBER ) ,
            Property(name="last_update"  ,data_type=DataType.DATE ),
            
            ])

else :
    collection=client.collections.get("example_collection")
    

In [ ]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

In [ ]:
from tqdm import tqdm
with collection.batch.fixed_size(batch_size=1 , concurrent_requests=1)as batch :
    for document in tqdm(data):
        uuid=generate_uuid5(document)
        batch.add_object(
            properties=document ,
            uuid=uuid
        )

In [ ]:
results =collection.query.near_text(query="america" , limit=2)

In [ ]:
for result in results.objects :
    print_object_properties(result.properties)
        

In [ ]:
results=collection.query.fetch_objects(filters=Filter.by_property("user_ratings").greater_or_equal(4.6) , limit=3)

In [ ]:
len (results.objects)

In [ ]:
results=collection.query.near_text(limit=2 , query=" califor" , filters=Filter.by_property("budget").contains_any(["low" ,"moderate"] ) )

In [ ]:
results=collection.query.bm25(query="california" ,limit=2 )

In [ ]:
import flask_app
from weaviate.classes.query import Rerank

In [ ]:
results=collection.query.hybrid(query="california vacation" ,filters=Filter.by_property("budget").contains_any(["low","moderate"]) ,alpha=0.1  ,limit=4 ,
                               rerank=Rerank(
                                   query="fun place" ,
                                   prop="attractions"
                                   
                               ) )

In [ ]:
from weaviate.classes.query import Rerank

response = collection.query.near_text(
    query="'I want suggestions to travel during Winter. I want cheap and fun places.'",  
    limit=5,
    rerank=Rerank(query="aaaa" ,
        prop="attractions"             
    )
)


In [ ]:
len(response.objects )

In [ ]:
import os
import sys

In [ ]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))

In [ ]:
if '__file__' in globals():
    # When imported as a module
    BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
else:
    # When run interactively (Jupyter, shell, etc.)
    # Assumes the script is executed from the project root or one level up
    if os.path.exists("data") and os.path.exists("app"):
        BASE_DIR = os.getcwd()
    else:
        # Try to go up one level
        parent = os.path.dirname(os.getcwd())
        if os.path.exists(os.path.join(parent, "data")) and os.path.exists(os.path.join(parent, "app")):
            BASE_DIR = parent
        else:
            BASE_DIR = os.getcwd()  # fallback

In [ ]:
BASE_DIR

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter